# Data formtting

In [1]:
import pandas as pd

In [6]:
data_df = pd.read_csv("../data/test/qwen2_5/08-04-26/results_traversal_ck_300_t_01 copy.csv")
data_df.shape

(623, 15)

In [7]:
# drop the columns that are not needed for analysis
columns_to_drop = ['is_supported', 'confidence', 'confidence', 'supporting_evidence', 'reasoning']
data_df = data_df.drop(columns=columns_to_drop)
data_df.head()

,doc_id,document,summary,generated_summary,document_triplets,gen_summary_triplets,summary_triplet_subject,summary_triplet_predicate,summary_triplet_object,summary_triplet_full,fact_eval_res_lora
0,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","More than 1,000 homeless people",referred to,charity,"['More than 1,000 homeless people', 'referred ...","{""summary_triple"": [""More than 1,000 homeless ..."
1,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...",charity,located in,Wales,"['charity', 'located in', 'Wales']","{""summary_triple"": [""More than 1,000 homeless ..."
2,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...",referral,occurred in,last year,"['referral', 'occurred in', 'last year']","{""summary_triple"": [""charity"", ""located in"", ""..."
3,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","More than 1,000 homeless people",associated with,Wales,"['More than 1,000 homeless people', 'associate...","{""summary_triple"": [""referral"", ""occurred in"",..."
4,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...",charity,associated with,last year,"['charity', 'associated with', 'last year']","{""summary_triple"": [""More than 1,000 homeless ..."


In [8]:
# Check the structure of fact_eval_res column
print(data_df['fact_eval_res_lora'].iloc[0])
print(type(data_df['fact_eval_res_lora'].iloc[0]))

{"summary_triple": ["More than 1,000 homeless people", "referred to", "charity"], "is_supported": true, "supporting_evidence": [["Prison Link Cymru", "had", "1,099 referrals"]], "confidence": 1, "reasoning": "The Text Triplets explicitly mention Prison Link Cymru had 1,099 referrals - this directly supports that More than 1,000 homeless people were referred to Charity."}
<class 'str'>


In [9]:
import json
import ast

# Re-load the original data
# data_df = pd.read_csv('../data/test/lora/results_traversal_lora.csv')

def parse_fact_eval(s):
    """Parse fact_eval_res with multiple fallback strategies"""
    # Strategy 1: Fix double braces and extra brackets, try JSON
    s_fixed = s.replace('{{', '{').replace('}}', '}')
    if s_fixed.endswith('}]}]'):
        s_fixed = s_fixed[:-2]
    
    try:
        return json.loads(s_fixed)
    except:
        pass
    
    # Strategy 2: Try using ast.literal_eval (handles single quotes)
    try:
        # First fix the double braces and extra brackets
        s_fixed = s.replace('{{', '{').replace('}}', '}')
        if s_fixed.endswith('}]}]'):
            s_fixed = s_fixed[:-2]
        
        # Replace true/false/null with Python equivalents for ast.literal_eval
        s_fixed = s_fixed.replace('true', 'True').replace('false', 'False').replace('null', 'None')
        return ast.literal_eval(s_fixed)
    except:
        pass
    
    return None

# Expand the fact_eval_res column - create one row per evaluation item
rows = []
parse_errors = []

for idx, row in data_df.iterrows():
    try:
        # Parse the JSON string
        parsed = parse_fact_eval(str(row['fact_eval_res_lora']))
        
        # Handle both dict and list cases
        if parsed:
            # If it's a single dict, wrap it in a list
            if isinstance(parsed, dict):
                parsed = [parsed]
            
            # If it's a list with items, create a row for each item
            if isinstance(parsed, list) and len(parsed) > 0:
                for eval_result in parsed:
                    new_row = row.to_dict()
                    new_row['summary_triple'] = str(eval_result.get('summary_triple', None))
                    new_row['is_supported'] = eval_result.get('is_supported', None)
                    new_row['supporting_evidence'] = str(eval_result.get('supporting_evidence', None))
                    new_row['confidence'] = eval_result.get('confidence', None)
                    new_row['reasoning'] = eval_result.get('reasoning', None)
                    rows.append(new_row)
            else:
                # Empty list case
                parse_errors.append({
                    'idx': idx, 
                    'fact_eval_res': str(row['fact_eval_res_lora'])
                })
                new_row = row.to_dict()
                new_row['summary_triple'] = None
                new_row['is_supported'] = None
                new_row['supporting_evidence'] = None
                new_row['confidence'] = None
                new_row['reasoning'] = None
                rows.append(new_row)
        else:
            # If parsing fails or list is empty, keep original row with None values
            parse_errors.append({
                'idx': idx, 
                'fact_eval_res': str(row['fact_eval_res_lora'])
            })
            new_row = row.to_dict()
            new_row['summary_triple'] = None
            new_row['is_supported'] = None
            new_row['supporting_evidence'] = None
            new_row['confidence'] = None
            new_row['reasoning'] = None
            rows.append(new_row)
    except Exception as e:
        # If parsing fails, keep original row with None values and log the error
        parse_errors.append({
            'idx': idx, 
            'error': str(e),
            'fact_eval_res': str(row['fact_eval_res_lora'])
        })
        new_row = row.to_dict()
        new_row['summary_triple'] = None
        new_row['is_supported'] = None
        new_row['supporting_evidence'] = None
        new_row['confidence'] = None
        new_row['reasoning'] = None
        rows.append(new_row)

# Create new dataframe with expanded rows
data_df = pd.DataFrame(rows)

print(f"Total rows: {len(data_df)}")
print(f"Parse errors: {len(parse_errors)}")
print(f"Rows with is_supported=True: {(data_df['is_supported'] == True).sum()}")
print(f"Rows with is_supported=False: {(data_df['is_supported'] == False).sum()}")
print(f"Rows with is_supported=None: {data_df['is_supported'].isna().sum()}")
print(f"\nValue counts:")
print(data_df['is_supported'].value_counts(dropna=False))

if parse_errors:
    print(f"\nFirst few remaining errors:")
    for i, error in enumerate(parse_errors):
        print(f"\nError {i+1} at row {error['idx']}:")
        print(f"  Data: {error['fact_eval_res']}")
        if 'error' in error:
            print(f"  Error: {error['error']}")

# Display the result
# data_df.tail(10)

Total rows: 623
Parse errors: 0
Rows with is_supported=True: 391
Rows with is_supported=False: 232
Rows with is_supported=None: 0

Value counts:
is_supported
True     391
False    232
Name: count, dtype: int64


In [7]:
updated_json =  {"summary_triple": ["Inditex", "reported", "rise in profits"], "is_supported": True, "supporting_evidence": [["world's biggest clothing retailer", "posted net earnings of €1.26bn", "sales jumped from €9.4bn"], ["net earnings of €1.26bn", "up by 8%", "same period last year"]], "confidence": 2, "reasoning": "Direct support: 'world's biggest clothing retailer' refers to Inditex, posted net earnings showing an 8% rise"}



In [8]:
# find a row by index
idx = 451
data_df.at[data_df.index[idx], 'fact_eval_res_lora'] = json.dumps(updated_json)
print(data_df.at[data_df.index[idx], 'fact_eval_res_lora'])

{"summary_triple": ["Inditex", "reported", "rise in profits"], "is_supported": true, "supporting_evidence": [["world's biggest clothing retailer", "posted net earnings of \u20ac1.26bn", "sales jumped from \u20ac9.4bn"], ["net earnings of \u20ac1.26bn", "up by 8%", "same period last year"]], "confidence": 2, "reasoning": "Direct support: 'world's biggest clothing retailer' refers to Inditex, posted net earnings showing an 8% rise"}


In [10]:
# save the expanded dataframe to a new CSV file
# data_df.to_csv('../data/test/qwen3_quantized/results_traversal_qwen3_8b_t_01.csv', index=False)
data_df.to_csv('../data/test/qwen2_5/08-04-26/results_traversal_ck_300_t_01_expanded.csv', index=False)